# Хроматическое число $\mathbb{R}^4$

### Описание алгоритма

Алгоритм (подробное обоснование — в `docs/article.tex`):

- строим решётку (`grid`) и соответствующее ей разбиение Вороного (`VoronoiPolyhedra`)
- ищем все многогранники центрального региона и его диаметр
- составляем список граней центрального многогранника (`list_faces`)
- перебираем различные подрешётки; для каждой ищем приведённый базис (LLL), находим длину минимального вектора и минимальное расстояние между векторами и их комбинациями
- ищем точку $s$ — середину отрезка между точкой $(0, 0, 0, 0)$ и центром ближайшего многогранника
- находим расстояние от точки $s$ до центрального многогранника
- умножаем это расстояние на 2 и получаем расстояние между многогранниками (центральным и ближайшим к нему многогранником из подрешётки); по лемме Иванова минимальное расстояние реализуется в точках, симметричных относительно $s$
- определитель матрицы перехода $k = |\det(M)|$ — количество цветов раскраски, найденное нормированное расстояние $d = 2D/\mathrm{diam}(V_0)$ — запрещённое расстояние
- для каждого определителя выбираем матрицу с максимальным $d$; раскраска пригодна при $d \geq 1$

### Основные структуры данных `VoronoiPolyhedra`

- `central` — список вершин центрального многогранника
- `coords4` — список координат центров многогранников Вороного

- `vertices` — список координат вершин разбиения Вороного
- `regions` — список индексов координат вершин многогранников разбиения Вороного (в `vertices`)
- `ridge_vertices` — вершины 3-мерных граней разбиения Вороного через их индексы в `vertices`
- `ridge_points` — список пар центров многогранников Вороного, между которыми есть общая грань, через индексы в `coords4` (по сути — соседние многогранники)

- `central_region_index` — индексы вершин центрального региона (в `vertices`)
- `faces_3d` — 3-мерные грани центрального многогранника в индексах вершин
- `edge_central_coords` — список списков вершин каждой 3-мерной грани центрального многогранника
- `faces_2d` — 2-мерные грани с разбивкой по 3-мерным граням (в индексах)
- `list_faces` — 3-мерные грани с разбивкой на 2-мерные (в координатах)
- `list_edges` — общий список рёбер (пары индексов вершин)

- `list_neigh_points` — координаты центров многогранников, граничащих с центральным (по сути — векторы нормали к 3-мерным граням центрального многогранника)
- `v_norm` — нормированные векторы нормалей (в том же порядке)
- `vertex_to_faces` — для каждой вершины центрального региона список содержащих её 3-мерных граней
- `polyhedrons` — объекты `Face3D` (грани с нормалями, 2-мерными гранями и рёбрами)
- `delaunay` — триангуляция Делоне центрального многогранника
- `max_len` — диаметр центрального многогранника $\mathrm{diam}(V_0)$

### Построение многогранника Вороного

In [ ]:
# подключаем пакет из src/ (альтернатива: pip install -e .)
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np

from voronoi4d import VoronoiPolyhedra

grid = np.array([
    [2, 0, 0, 0],
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [1, 0, 0, 1]
], dtype=float)

vor4 = VoronoiPolyhedra(grid)
vor4.build()

In [ ]:
# вершины диаграммы Вороного
vor4.vertices

In [ ]:
# регионы
vor4.regions[1]

In [ ]:
# список конечных областей Вороного (-1 в индексах означает, что область бесконечна)
regions_finite = [r for r in vor4.regions if -1 not in r]
print(len(regions_finite))

In [ ]:
# максимальный размер региона (по числу вершин)
len_r_max = max(len(r) for r in vor4.regions)
count_r = sum(1 for r in vor4.regions if len(r) == len_r_max)

print('количество вершин =', len_r_max, 'количество =', count_r)

In [ ]:
# центральный многогранник: вершины и диаметр
print('число вершин центрального региона =', len(vor4.central))
print('диаметр diam(V0) =', vor4.max_len)
vor4.central

In [ ]:
# 3-мерные грани центрального многогранника и их нормали
for i, poly in enumerate(vor4.polyhedrons):
    print(f'грань {i}: вершин={len(poly.vertices)}, 2D-граней={len(poly.faces)}, нормаль={poly.normal}')